In [114]:
# This notebook will cover part 1.13 in the llm-zoomcamp course, I need to create this
# separate because in the main llm-zoomcamp files,
# I have turned search into database rather than minsearch, and it is much
# simpler to illustrate turning RAG into agentic RAG using minsearch. 

# The first half of this notebook will be just the older minsearch stuff, I will create a divider 
# below from when the agentic code starts.

In [115]:
from openai import OpenAI
openai_client = OpenAI()

In [116]:
from dotenv import load_dotenv
load_dotenv()

True

In [117]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [118]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Yes — in most cases you can still join after a course has started, but it depends on the course policy and how far along it is.

A few common possibilities:
- **Self-paced course:** usually yes, you can join anytime.
- **Live cohort course:** maybe, if the instructor allows late enrollment.
- **University/classroom course:** depends on add/drop deadlines.

If you want, I can help you draft a quick message to the instructor or support team asking whether late enrollment is possible.


In [119]:
context = """I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active."""

In [120]:
prompt = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
{question}

Context:
{context}'''

In [121]:
print(prompt)

Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
I just discovered the course. Can I join now?

Context:
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors

In [122]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.


In [123]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [124]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [125]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [126]:
"""
We start with R in RAG, which is Retrieval, but he calls it SEARCH.
"""

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [127]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
index.search(question)  # returns questions from all of the courses, llm_zoomcamp, AI Dev Zoomcamp, Data Engineering Zoomcamp, etc.

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [128]:
index.search(question, filter_dict={'course':'llm-zoomcamp'})  # returns all the questions only from the llm-zoomcamp course

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [129]:
search_results =index.search(question, filter_dict={'course':'llm-zoomcamp'}, num_results=5) # if we just want 5 results

In [130]:
# this was our original RAG function
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

# We have now covered the search function, so let's add it
def search(query, course='llm-zoomcamp'):
    boost_dict={'question':2.0, 'section':0.5}
    filter_dict={'course':course}
    
    return index.search(query,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [131]:
search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [132]:
"""
We move on to the A in RAG, which is Augmented, but he calls it PROMPT.


He said that Prompt is split into two parts: 

1) Instructions 
2) User prompt

So let's split up what we had towards the beginnig of all of this.
"""

"\nWe move on to the A in RAG, which is Augmented, but he calls it PROMPT.\n\n\nHe said that Prompt is split into two parts: \n\n1) Instructions \n2) User prompt\n\nSo let's split up what we had towards the beginnig of all of this.\n"

In [133]:
INSTRUCTIONS = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"
'''

In [134]:
USER_PROMPT_TEMPLATE = '''
Question: 
{question}

Context:
{context}
'''

In [135]:
# Next up, we turn the python dictionaries that we got from our search into one long string, this makes it
# easier for the LLM to read and understand.
# I'll copy over a function that I wrote before, it takes the search_results which is the two json dicitonaries 
# and it turns them into one long string. 

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["course"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip() 

build_context(search_results)



'llm-zoomcamp\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nllm-zoomcamp\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nllm-zoomcamp\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nllm-zoomcamp\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chain

In [136]:
# Then we feed this long string into our prompt template that we saw before, that we will send to the llm. 
# Say search_result we got from part one, build_context that turns it into a long string 
# and the prompt, which simply formats the long string using the USER_PROMPT_TEMPLATE. 
# And we remove any trailing or leading whitespace with strip. 
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)

print(prompt)

Question: 
I just discovered the course. Can I join now?

Context:
llm-zoomcamp
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

llm-zoomcamp
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

llm-zoomcamp
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

llm-zoomcamp
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, 

In [137]:
"""
Then the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, 
hence generation.
"""

'\nThen the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, \nhence generation.\n'

In [138]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [139]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [140]:
# We can now put this together into an updated llm function.
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [141]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [142]:
# ---------------------------------------------
# The above all minsearch RAG stuff we have seen before, below is agentic code

In [143]:
# When you have a typo in Ollama and you send it into our lexiographical text minsearch function
rag('how do I run Olama?')

'I do not know'

In [144]:
# When you send it directly to ChatGPT, i.e. call the llm directly with this 
search('how do I run Olama?')   

[{'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that decide which tools to call. The goal is to see how AI fits into production-style workflows, not just notebook cells.\n\nKestra's AI plugins also work with any major provider (OpenAI, Gemini, Anthropic, and more), so you can swap providers in a flow without changing anything else. See the [module intro](https://github.com/DataTalksClub/llm-zoom

In [145]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [146]:
messages = [
        {"role": "user", "content": "I just discovered the course. Can I join it?"}
    ]

In [147]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

In [148]:
# Notice how the llm has slightly changed the wording of the question? - this is what we want 
# because if we have a typo very likely it will correct it for us!
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment join course discovered course"}', call_id='call_9VIaCK8kwf53MPV21Ojj0np1', name='search', type='function_call', id='fc_0b6caf95bc591ad8006aba8178f00487d2acc644cc86ffcc2e', async_=None, caller=None, namespace=None, status='completed')]

In [149]:
# So we assign the output of the llm to 'call'.
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment join course discovered course"}', call_id='call_9VIaCK8kwf53MPV21Ojj0np1', name='search', type='function_call', id='fc_0b6caf95bc591ad8006aba8178f00487d2acc644cc86ffcc2e', async_=None, caller=None, namespace=None, status='completed')

In [150]:
# so call.arguments points at the query that the llm has 'changed' our initial prompt to!
call.arguments

'{"query":"Can I join the course after it has started? late enrollment join course discovered course"}'

In [151]:
import json 

args = json.loads(call.arguments)

In [152]:
results = search(**args)

In [153]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub

In [154]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomc

In [155]:
# We format the above output in a way such that we can plug it into the API call to OpenAI further down.

# IMPORTANTLY, function_call_output is my program's REPLY to the request the model made. 
# The name is a bit confusing, because "output" here refers to the output of the function, 
# i.e. what search() returned. It is NOT the llm's output.

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [156]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [157]:
# Note call is the response from the first call we 
# made to the llm
messages.append(call)

In [158]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment join course discovered course"}', call_id='call_9VIaCK8kwf53MPV21Ojj0np1', name='search', type='function_call', id='fc_0b6caf95bc591ad8006aba8178f00487d2acc644cc86ffcc2e', async_=None, caller=None, namespace=None, status='completed')]

In [159]:
messages.append(function_call_output)

In [160]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment join course discovered course"}', call_id='call_9VIaCK8kwf53MPV21Ojj0np1', name='search', type='function_call', id='fc_0b6caf95bc591ad8006aba8178f00487d2acc644cc86ffcc2e', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_9VIaCK8kwf53MPV21Ojj0np1',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certific

In [161]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

print(response.output_text)

Yes — you can still join. The course materials are available, so you can start anytime.

If you want a certificate, though, you’ll need to submit your project while submissions are still open.


In [162]:
#------- The above is everything from part 1.13 making out RAG agentic 
# From here on we do part 1.14 which is creating the agentic loop

In [163]:
# We start by telling the llm that it can loop, but in human words, we set this in 
# the instructions:

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [164]:
# Then we add the instructions to our messages; if you look above, we only had the 
# user role. Now we add developer. 
question = 'I just discovered this course. Can I join it?'

messages = [
    {"role":"developer", "content": question},  # According to Claude it matters which way round, developer first then user
        {"role": "user", "content": instructions}
    ]

In [165]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

In [ ]:
# Note there are two (maybe three) items when you print here. 
# The agent took out one question and produced 2,3, could even be 4 or more 'guesses' 
# it can change depending on the run.
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course enrollment can I join this course discovered course"}', call_id='call_hawv7QcUpS1ssWrg4jZeO2CJ', name='search', type='function_call', id='fc_005546db024198ee006aba817c385c87d2ae6ea650678c0a66', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course registration eligibility join course enrollment new student"}', call_id='call_PwVEzDjEQ5adhTp7oKXQidSk', name='search', type='function_call', id='fc_005546db024198ee006aba817c387487d28f9e3fac3b228b8c', async_=None, caller=None, namespace=None, status='completed')]

In [ ]:
for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"query":"join course enrollment can I join this course discovered course"}', call_id='call_hawv7QcUpS1ssWrg4jZeO2CJ', name='search', type='function_call', id='fc_005546db024198ee006aba817c385c87d2ae6ea650678c0a66', async_=None, caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"course registration eligibility join course enrollment new student"}', call_id='call_PwVEzDjEQ5adhTp7oKXQidSk', name='search', type='function_call', id='fc_005546db024198ee006aba817c387487d28f9e3fac3b228b8c', async_=None, caller=None, namespace=None, status='completed')
